# 3. App Registrations and Managed Identities

AZ-500 expects you to configure OAuth flows, manage consent, and implement managed identities. This builds directly on the [Azure Authentication lab](../../../enterprise-patterns/azure-authentication/).

## Setup

```bash
cd security/az-500/01-identity-and-access
docker compose up -d
uv sync
```

## App registration checklist

When creating an app registration in the exam, consider:

| Setting | Where to configure | Purpose |
|---------|-------------------|----------|
| **Display name** | Overview | Human-readable identifier |
| **Supported account types** | Authentication | Single tenant, multi-tenant, or personal |
| **Redirect URIs** | Authentication | Where tokens are sent after auth |
| **Client secret or certificate** | Certificates & secrets | Prove identity for confidential clients |
| **API permissions** | API permissions | What resources the app can access |
| **Expose an API** | Expose an API | Define scopes other apps can request |
| **App roles** | App roles | Application-level permissions (no user) |

In [ ]:
import httpx, json, base64

ENTRA = 'http://localhost:9000/contoso'
TOKEN_URL = f'{ENTRA}/oauth2/v2.0/token'

def decode(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# Demonstrate the difference: delegated (scp) vs application (roles) permissions

# 1. Delegated: user signs in, app acts on behalf of user
print('=== 1. Delegated permission (OAuth scope) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-b/Files.Read',
})
delegated = decode(r.json()['access_token'])
print(f'Token type: delegated (has "scp" claim)')
print(f'  scp (scopes): {delegated.get("scp")}')
print(f'  upn (user):   {delegated.get("upn")}')
print(f'  aud:          {delegated.get("aud")}')

# 2. Application: app calls on its own behalf (no user)
print('\n=== 2. Application permission (app role) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
app_only = decode(r.json()['access_token'])
print(f'Token type: application (has "roles" claim, no user)')
print(f'  roles:        {app_only.get("roles")}')
print(f'  upn:          {app_only.get("upn", "(none — app-only token)")}')
print(f'  aud:          {app_only.get("aud")}')

## OAuth consent types

| Type | Who approves | When |
|------|-------------|------|
| **User consent** | Individual user | Low-risk delegated permissions (e.g., `User.Read`) |
| **Admin consent** | Tenant admin | High-risk delegated or any application permissions |
| **Pre-authorized** | App owner | First-party apps or verified publishers |

### Managing consent

```bash
# Grant admin consent for an app
az ad app permission admin-consent --id <app-id>

# View current permissions
az ad app permission list --id <app-id>

# Configure user consent settings (restrict what users can consent to)
az rest --method PATCH \
  --uri 'https://graph.microsoft.com/v1.0/policies/authorizationPolicy' \
  --body '{"defaultUserRolePermissions": {"permissionGrantPoliciesAssigned": ["managePermissionGrantsForSelf.microsoft-user-default-low"]}}'
```

### Exam tip: consent risk
A common attack vector is **illicit consent grants** — an attacker tricks a user into granting an app `Mail.Read` or `Files.ReadWrite.All`. AZ-500 expects you to know how to:
1. Restrict user consent to verified publishers only.
2. Require admin consent for high-risk permissions.
3. Review existing grants with `az ad app permission list`.

---
## Managed identities — implementation

### System-assigned vs user-assigned — when to use which

| | System-assigned | User-assigned |
|-|----------------|---------------|
| **Lifecycle** | Created/deleted with the resource | Independent resource |
| **Sharing** | One resource only | Attach to many resources |
| **Use case** | Single-purpose workload | Shared identity across VMs, apps |
| **Blue/green** | New identity per slot | Same identity across slots |

### Azure CLI commands

In [ ]:
# Azure CLI reference for managed identities
cli_commands = {
    'Enable system-assigned MI on a VM': 
        'az vm identity assign -g rg-prod -n my-vm',
    'Create user-assigned MI':
        'az identity create -g rg-prod -n my-app-identity',
    'Assign user-assigned MI to Container App':
        'az containerapp identity assign -g rg-prod -n my-app --user-assigned /subscriptions/.../my-app-identity',
    'Grant MI access to Key Vault secrets':
        'az role assignment create --assignee <MI-principal-id> --role "Key Vault Secrets User" --scope /subscriptions/.../resourceGroups/rg-prod/providers/Microsoft.KeyVault/vaults/my-kv',
    'Grant MI access to Storage blobs':
        'az role assignment create --assignee <MI-principal-id> --role "Storage Blob Data Contributor" --scope /subscriptions/.../resourceGroups/rg-prod/providers/Microsoft.Storage/storageAccounts/mysa',
    'Test MI from inside a VM (IMDS)':
        'curl "http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=https://vault.azure.net" -H Metadata:true',
}

print('=== Managed Identity CLI Reference ===\n')
for desc, cmd in cli_commands.items():
    print(f'# {desc}')
    print(f'{cmd}\n')

### IMDS (Instance Metadata Service)

Inside Azure VMs/containers, managed identities get tokens from IMDS at `169.254.169.254`. The Azure SDK calls this automatically via `DefaultAzureCredential`.

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# This works on your laptop (az login), in CI (env vars), and in Azure (managed identity)
credential = DefaultAzureCredential()
client = SecretClient(vault_url='https://my-kv.vault.azure.net', credential=credential)
secret = client.get_secret('db-password')
```

### Exam tip
- Always prefer **managed identity** over stored secrets.
- For AKS: use **workload identity** (federated) instead of pod-mounted secrets.
- `DefaultAzureCredential` is the recommended credential class for all scenarios.

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **App registration** | Client ID, secret/cert, permissions, exposed scopes, app roles |
| **Delegated vs application** | `scp` claim (user present) vs `roles` claim (app-only) |
| **Consent management** | Restrict user consent, require admin consent for risky permissions |
| **System-assigned MI** | 1:1 with resource, auto-deleted |
| **User-assigned MI** | Reusable across resources, manual lifecycle |
| **DefaultAzureCredential** | One credential class for all environments |

**Next lab**: [02 — Secure Networking](../../02-networking/)